# Normalization-Fix SSCD Evaluation

Evaluate the finished normalization-fix jobs:

- `nf_u128_n500_e100`
- `nf_u128_n500_e80`
- `nf_u256_n500_e100`

The notebook checks job logs, discovers configs and generated sample files, computes physics diagnostics, and runs SSCD near-copy/generalization evaluation.

## Imports

In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import subprocess
import sys
from glob import glob
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

for candidate in (PROJECT_DIR, PROJECT_DIR / "cosmo_diffusion"):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from simdiff_eval.io import as_nchw, load_npy, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, nearest_neighbor_distances, power_spectrum_summary
from simdiff_eval.sscd import load_sscd_torchscript, sscd_embeddings, sscd_generalization_metrics

PROJECT_DIR


## Run Configuration

In [ ]:
RUN_SPECS = [
    {"label": "U128 n=500 e=100", "run_base": "nf_u128_n500_e100", "arch": "u128", "n_train": 500, "epochs": 100},
    {"label": "U128 n=500 e=80", "run_base": "nf_u128_n500_e80", "arch": "u128", "n_train": 500, "epochs": 80},
    {"label": "U256 n=500 e=100", "run_base": "nf_u256_n500_e100", "arch": "u256", "n_train": 500, "epochs": 100},
]

LOG_DIR = PROJECT_DIR / "logs" / "normalization_fixes"
OUTPUT_DIR = PROJECT_DIR / "results" / "normalization_fixes_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add scratch/output roots here, or set before starting Jupyter:
# export NF_SEARCH_ROOTS=/path/to/run/root:/another/run/root
EXTRA_SEARCH_ROOTS = [
    Path(p).expanduser()
    for p in os.environ.get("NF_SEARCH_ROOTS", "").split(":")
    if p.strip()
]
for env_name in ("SCRATCH", "WORK", "COSMODIFF_RUN_ROOT"):
    if os.environ.get(env_name):
        EXTRA_SEARCH_ROOTS.append(Path(os.environ[env_name]).expanduser())

SEARCH_ROOTS = [
    PROJECT_DIR / "local",
    PROJECT_DIR / "results",
    PROJECT_DIR / "outputs",
    PROJECT_DIR / "checkpoints",
    *EXTRA_SEARCH_ROOTS,
]
SEARCH_ROOTS = [p for p in dict.fromkeys(SEARCH_ROOTS) if p.exists()]

# Optional direct overrides. Fill these in if discovery misses files.
# Example:
# MANUAL_RUN_PATHS = {
#     "nf_u128_n500_e100": {
#         "config_path": "/path/to/nf_u128_n500_e100.yaml",
#         "checkpoint_path": "/path/to/nf_u128_n500_e100/checkpoint-50000",
#         "sample_path": "results/tables/samples/nf_u128_n500_e100_seed123.npy",
#     },
# }
MANUAL_RUN_PATHS: dict[str, dict[str, str]] = {}

SSCD_PATH = Path(os.environ.get("SSCD_PATH", Path.home() / ".cache" / "torch" / "hub" / "sscd_disc_mixup.torchscript.pt"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 123
NUM_SAMPLES_IF_MISSING = 64
SAMPLE_BATCH_SIZE = 8
IMAGE_SIZE_BY_ARCH = {"u64": 128, "u128": 128, "u256": 128}

# Keep these capped for interactive notebook work. Loading all 500 raw cubes can kill the kernel.
MAX_REAL_RAW_CUBES_TO_LOAD = 32
MAX_REAL_PHYSICS = 512
MAX_GENERATED_PHYSICS = 64
MAX_REAL_NN = 512
MAX_GENERATED_NN = 64
MAX_REAL_SSCD = 512
MAX_GENERATED_SSCD = 64
SSCD_BATCH_SIZE = 32
SSCD_THRESHOLD = 0.6
SSCD_THRESHOLDS = [0.4, 0.5, 0.6, 0.7]
SSCD_RENDER_MODE = "fixed"

print("project:", PROJECT_DIR)
print("logs:", LOG_DIR)
print("outputs:", OUTPUT_DIR)
print("search roots:")
for root in SEARCH_ROOTS:
    print("  ", root)
print("sscd:", SSCD_PATH, "exists=", SSCD_PATH.exists())
print("device:", DEVICE)


## Discover Logs, Configs, Samples, and Checkpoints

If a path is missing, edit the candidate glob patterns in `discover_run_files`.

In [ ]:
def glob_paths(patterns: list[Path]) -> list[Path]:
    paths: list[Path] = []
    for pattern in patterns:
        try:
            paths.extend(Path(p) for p in glob(str(pattern), recursive=True))
        except OSError as exc:
            print(f"Skipping {pattern}: {exc}")
    return sorted(set(paths))


def first_existing(patterns: list[Path]) -> Path | None:
    matches = [p for p in glob_paths(patterns) if p.exists()]
    return matches[0] if matches else None


def clean_path_token(token: str) -> Path:
    token = token.strip().strip("'\"`,;:()[]{}<>")
    return Path(token).expanduser()


def log_path_candidates(log_files: list[Path], base: str | None = None) -> list[Path]:
    candidates: list[Path] = []
    for log_file in log_files:
        text = log_file.read_text(errors="replace")
        for match in re.findall(r"(?:/|~)[^\s'\"<>]+", text):
            path = clean_path_token(match)
            if base is None or base in str(path):
                candidates.append(path)
    return sorted(set(candidates))


def log_config_candidates(log_files: list[Path], base: str) -> list[Path]:
    candidates: list[Path] = []
    for log_file in log_files:
        text = log_file.read_text(errors="replace")
        for pattern in (
            r"--config\s+([^\s'\"<>]+)",
            r"CONFIG_PATH=([^\s'\"<>]+)",
            r"config(?:_path)?[=:]\s*([^\s'\"<>]+\.ya?ml)",
            r"(?:/|~)[^\s'\"<>]+\.ya?ml",
        ):
            for match in re.findall(pattern, text):
                path = clean_path_token(match)
                if path.exists() and path.suffix in {".yaml", ".yml"}:
                    candidates.append(path)
    base_matches = [p for p in candidates if base in str(p)]
    return sorted(set(base_matches or candidates))


def is_checkpoint_like(path: Path) -> bool:
    if not path.exists():
        return False
    if path.name == "model_index.json":
        return True
    if path.is_dir() and ((path / "model_index.json").exists() or any(path.glob("checkpoint-*"))):
        return True
    if path.is_dir() and path.name.startswith("checkpoint-"):
        return True
    return path.suffix in {".pt", ".pth"}


def checkpoint_sort_key(path: Path) -> tuple[int, float, str]:
    match = re.search(r"checkpoint-(\d+)", str(path))
    step = int(match.group(1)) if match else -1
    try:
        mtime = path.stat().st_mtime
    except OSError:
        mtime = 0.0
    return step, mtime, str(path)


def choose_checkpoint(candidates: list[Path]) -> Path | None:
    existing = [p.parent if p.name == "model_index.json" else p for p in candidates if is_checkpoint_like(p)]
    if not existing:
        return None
    return sorted(existing, key=checkpoint_sort_key)[-1]


def discover_run_files(spec: dict[str, Any]) -> dict[str, Any]:
    base = spec["run_base"]
    log_files = glob_paths([LOG_DIR / f"{base}_*.err", LOG_DIR / f"{base}_*.out"])
    log_candidates = log_path_candidates(log_files, base)
    config_candidates_from_logs = log_config_candidates(log_files, base)

    config_patterns = [
        PROJECT_DIR / "local" / "normalization_fixes" / "configs" / f"{base}.yaml",
        PROJECT_DIR / "local" / "normalization_fixes" / "configs" / f"{base}*.yaml",
        PROJECT_DIR / "configs" / "normalization_fixes" / f"{base}.yaml",
        PROJECT_DIR / "configs" / "normalization_fixes" / f"{base}*.yaml",
        PROJECT_DIR / "local" / "**" / f"{base}*.yaml",
        PROJECT_DIR / "configs" / "**" / f"{base}*.yaml",
    ]
    sample_patterns = [
        PROJECT_DIR / "results" / "tables" / "samples" / f"{base}_seed{SEED}.npy",
        PROJECT_DIR / "results" / "tables" / "samples" / f"{base}*.npy",
        PROJECT_DIR / "results" / "normalization_fixes" / "samples" / f"{base}*.npy",
        PROJECT_DIR / "results" / "samples" / f"{base}*.npy",
        PROJECT_DIR / "local" / "normalization_fixes" / "samples" / f"{base}*.npy",
    ]
    checkpoint_patterns: list[Path] = []
    config_search_patterns: list[Path] = []
    sample_search_patterns: list[Path] = []
    for root in SEARCH_ROOTS:
        config_search_patterns.extend([
            root / "**" / f"*{base}*.yaml",
            root / "**" / f"*{base}*.yml",
        ])
        sample_search_patterns.extend([
            root / "**" / f"*{base}*.npy",
        ])
        checkpoint_patterns.extend([
            root / "**" / f"*{base}*" / "model_index.json",
            root / "**" / f"*{base}*" / "checkpoint-*",
            root / "**" / f"*{base}*",
        ])

    config = first_existing(config_patterns) or first_existing(config_search_patterns) or (config_candidates_from_logs[0] if config_candidates_from_logs else None)
    sample = first_existing(sample_patterns) or first_existing(sample_search_patterns)
    checkpoint_candidates = log_candidates + glob_paths(checkpoint_patterns)
    checkpoint = choose_checkpoint(checkpoint_candidates)

    manual = MANUAL_RUN_PATHS.get(base, {})
    if manual.get("config_path"):
        config = Path(manual["config_path"]).expanduser()
    if manual.get("sample_path"):
        sample = Path(manual["sample_path"]).expanduser()
        if not sample.is_absolute():
            sample = PROJECT_DIR / sample
    if manual.get("checkpoint_path"):
        checkpoint = Path(manual["checkpoint_path"]).expanduser()

    return {
        **spec,
        "n_logs": len(log_files),
        "latest_log": str(log_files[-1]) if log_files else None,
        "config_path": str(config) if config else None,
        "sample_path": str(sample) if sample else None,
        "checkpoint_path": str(checkpoint) if checkpoint else None,
        "config_exists": bool(config and config.exists()),
        "sample_exists": bool(sample and sample.exists()),
        "checkpoint_exists": bool(checkpoint and checkpoint.exists()),
        "n_config_candidates_from_logs": len(config_candidates_from_logs),
        "n_checkpoint_candidates": len([p for p in checkpoint_candidates if p.exists()]),
    }


run_table = pd.DataFrame([discover_run_files(spec) for spec in RUN_SPECS])
run_table

## Tail Finished Job Logs

In [ ]:
def tail_text(path: Path, n_lines: int = 80) -> str:
    lines = path.read_text(errors="replace").splitlines()
    return "\n".join(lines[-n_lines:])


for row in run_table.to_dict("records"):
    print("=" * 100)
    print(row["run_base"])
    latest_log = row.get("latest_log")
    if latest_log is None:
        print("No .err log discovered.")
        continue
    print(latest_log)
    print("-" * 100)
    print(tail_text(Path(latest_log), n_lines=80))

## Training Curves

Load saved training metrics from checkpoint/output directories and plot batch loss, epoch loss, and learning rate. If metrics are missing, this table will show which runs need log/checkpoint inspection.


In [ ]:
def config_output_dir(config_path: str | None) -> Path | None:
    if not config_path:
        return None
    import yaml
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    output_dir = cfg.get("io", {}).get("output_dir")
    return Path(output_dir).expanduser() if output_dir else None


def metrics_candidates(row: dict[str, Any]) -> list[Path]:
    candidates: list[Path] = []
    checkpoint = Path(row["checkpoint_path"]).expanduser() if row.get("checkpoint_path") else None
    output_dir = config_output_dir(row.get("config_path"))
    roots = []
    if checkpoint is not None:
        roots.extend([checkpoint, checkpoint.parent])
    if output_dir is not None:
        roots.append(output_dir)

    for root in roots:
        if root is None or not root.exists():
            continue
        if root.is_file() and root.suffix == ".json":
            candidates.append(root)
            continue
        candidates.extend(root.glob("metrics.json"))
        candidates.extend(root.glob("metrics_epoch_*.json"))
        candidates.extend(root.glob("checkpoint-epoch-*/metrics.json"))
    return sorted(set(candidates))


def read_metrics_json(path: Path) -> dict[str, Any] | None:
    try:
        metrics = json.loads(path.read_text())
    except Exception:
        return None
    if not isinstance(metrics, dict) or "epoch_loss" not in metrics:
        return None
    return metrics


def choose_metrics_file(paths: list[Path]) -> tuple[Path | None, dict[str, Any] | None]:
    best_path = None
    best_metrics = None
    best_key = (-1, -1.0)
    for path in paths:
        metrics = read_metrics_json(path)
        if metrics is None:
            continue
        key = (len(metrics.get("epoch_loss", [])), path.stat().st_mtime)
        if key > best_key:
            best_key = key
            best_path = path
            best_metrics = metrics
    return best_path, best_metrics


metrics_by_run: dict[str, dict[str, Any]] = {}
metrics_rows = []
for row in run_table.to_dict("records"):
    run_base = row["run_base"]
    path, metrics = choose_metrics_file(metrics_candidates(row))
    if metrics is None:
        metrics_rows.append({"run_base": run_base, "metrics_found": False, "metrics_path": None})
        continue
    metrics_by_run[run_base] = metrics
    epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
    epoch_lr = np.asarray(metrics.get("epoch_lr", []), dtype=float) if metrics.get("epoch_lr") is not None else np.array([])
    metrics_rows.append({
        "run_base": run_base,
        "metrics_found": True,
        "metrics_path": str(path),
        "n_epoch_loss": len(epoch_loss),
        "final_epoch_loss": float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        "min_epoch_loss": float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
        "final_lr": float(epoch_lr[-1]) if len(epoch_lr) else np.nan,
    })

metrics_table = pd.DataFrame(metrics_rows)
display(metrics_table)


In [ ]:
if metrics_by_run:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    for row in run_table.to_dict("records"):
        run_base = row["run_base"]
        label = row["label"]
        metrics = metrics_by_run.get(run_base)
        if metrics is None:
            continue

        loss = np.asarray(metrics.get("loss", []), dtype=float)
        if len(loss):
            window = max(1, min(200, len(loss) // 50 or 1))
            if window > 1:
                smooth = pd.Series(loss).rolling(window, min_periods=1).mean().to_numpy()
                axes[0].plot(smooth, label=label)
            else:
                axes[0].plot(loss, label=label)

        epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
        if len(epoch_loss):
            axes[1].plot(np.arange(len(epoch_loss)), epoch_loss, marker="o", ms=3, label=label)

        epoch_lr = np.asarray(metrics.get("epoch_lr", []), dtype=float) if metrics.get("epoch_lr") is not None else np.array([])
        if len(epoch_lr):
            axes[2].plot(np.arange(len(epoch_lr)), epoch_lr, marker="o", ms=3, label=label)

    axes[0].set_title("Batch loss")
    axes[0].set_xlabel("optimizer step")
    axes[0].set_ylabel("MSE loss")
    axes[0].set_yscale("log")
    axes[1].set_title("Epoch loss")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mean MSE loss")
    axes[1].set_yscale("log")
    axes[2].set_title("Learning rate")
    axes[2].set_xlabel("epoch")
    axes[2].set_ylabel("LR")
    axes[2].set_yscale("log")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    training_fig = OUTPUT_DIR / "normalization_fixes_training_curves.png"
    fig.savefig(training_fig, dpi=180)
    print("wrote", training_fig)
else:
    print("No metrics JSON files found. Check checkpoint paths or TensorBoard logs.")


## Generate Samples If Needed

Leave `RUN_SAMPLING = False` when you already have generated `.npy` files. If a sample path is missing but a checkpoint path exists, this cell prints the exact commands. Set `RUN_SAMPLING = True` only inside an interactive GPU job or a batch job.

In [ ]:
RUN_SAMPLING = False

sample_commands: list[list[str]] = []
for row in run_table.to_dict("records"):
    if row["sample_path"] is not None:
        continue
    if row["checkpoint_path"] is None:
        print(f"No sample or checkpoint found for {row['run_base']}.")
        print("  If checkpoints are on scratch, set NF_SEARCH_ROOTS and rerun the discovery cell.")
        print(f"  Example: export NF_SEARCH_ROOTS=/path/to/checkpoint/root")
        print(f"  Locator: find $PWD ${'{'}SCRATCH:-{'}'} -path '*{row['run_base']}*' -print | head -80")
        continue

    image_size = IMAGE_SIZE_BY_ARCH[row["arch"]]
    output = PROJECT_DIR / "results" / "tables" / "samples" / f"{row['run_base']}_seed{SEED}.npy"
    cmd = [
        sys.executable,
        "scripts/sample_cosmodiff.py",
        "--checkpoint", row["checkpoint_path"],
    ]
    if row.get("config_path"):
        cmd.extend(["--config", row["config_path"]])
    cmd.extend([
        "--output", str(output),
        "--num-samples", str(NUM_SAMPLES_IF_MISSING),
        "--batch-size", str(SAMPLE_BATCH_SIZE),
        "--image-size", str(image_size),
        "--seed", str(SEED),
        "--device", DEVICE,
    ])
    sample_commands.append(cmd)
    print(" ".join(shlex.quote(part) for part in cmd))

if RUN_SAMPLING:
    for cmd in sample_commands:
        subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    run_table = pd.DataFrame([discover_run_files(spec) for spec in RUN_SPECS])
    display(run_table)

## Load Real and Generated Samples

In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    if limit is None or len(arr) <= limit:
        return np.asarray(arr).copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return np.asarray(arr[idx]).copy()


data: dict[str, dict[str, Any]] = {}
load_rows = []
for row in run_table.to_dict("records"):
    base = row["run_base"]
    missing = []
    if row["config_path"] is None or not Path(row["config_path"]).exists():
        missing.append("config")
    if row["sample_path"] is None or not Path(row["sample_path"]).exists():
        missing.append("sample")
    if missing:
        load_rows.append({
            "run_base": base,
            "loaded": False,
            "reason": "missing " + " and ".join(missing),
            "config_path": row.get("config_path"),
            "sample_path": row.get("sample_path"),
            "checkpoint_path": row.get("checkpoint_path"),
        })
        continue

    real = load_real_from_config(row["config_path"], max_raw_samples=MAX_REAL_RAW_CUBES_TO_LOAD)
    generated = as_nchw(load_npy(row["sample_path"])).copy()
    data[base] = {"spec": row, "real": real, "generated": generated}
    load_rows.append({
        "run_base": base,
        "loaded": True,
        "real_shape": tuple(real.shape),
        "generated_shape": tuple(generated.shape),
        "max_real_raw_cubes_loaded": MAX_REAL_RAW_CUBES_TO_LOAD,
        "config_path": row["config_path"],
        "sample_path": row["sample_path"],
    })

load_table = pd.DataFrame(load_rows)
display(load_table)
if not data:
    raise RuntimeError(
        "No runs were loaded yet. Generate samples first, and make sure each row has an existing config_path. "
        "Use MANUAL_RUN_PATHS in the Run Configuration cell if auto-discovery cannot find them."
    )

## Real vs Generated Image Grids

Visual sanity check in normalized training space. These are not denormalized physical HI fields; they show what the diffusion model was trained to generate.


In [ ]:
def image_grid(images: np.ndarray, n: int = 6) -> np.ndarray:
    arr = np.asarray(images)
    if arr.ndim == 4:
        arr = arr[:, 0]
    arr = arr[:n]
    if len(arr) < n:
        pad = np.zeros((n - len(arr), *arr.shape[-2:]), dtype=arr.dtype)
        arr = np.concatenate([arr, pad], axis=0)
    return np.concatenate([arr[i] for i in range(n)], axis=1)


N_SHOW_IMAGES = 6
fig, axes = plt.subplots(len(data), 2, figsize=(12, 3.2 * len(data)), squeeze=False)
for row_idx, (base, bundle) in enumerate(data.items()):
    real = evenly_limit(bundle["real"], N_SHOW_IMAGES)
    generated = evenly_limit(bundle["generated"], N_SHOW_IMAGES)
    combined_values = np.concatenate([real.ravel(), generated.ravel()])
    vmin, vmax = np.nanpercentile(combined_values, [1, 99])

    axes[row_idx, 0].imshow(image_grid(real, N_SHOW_IMAGES), cmap="viridis", vmin=vmin, vmax=vmax)
    axes[row_idx, 0].set_title(f"{bundle['spec']['label']} real")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(image_grid(generated, N_SHOW_IMAGES), cmap="viridis", vmin=vmin, vmax=vmax)
    axes[row_idx, 1].set_title(f"{bundle['spec']['label']} generated")
    axes[row_idx, 1].axis("off")

fig.tight_layout()
image_grid_fig = OUTPUT_DIR / "normalization_fixes_real_generated_grids.png"
fig.savefig(image_grid_fig, dpi=180)
print("wrote", image_grid_fig)


## Physics Diagnostics

In [ ]:
physics_rows = []
for base, bundle in data.items():
    real = evenly_limit(bundle["real"], MAX_REAL_PHYSICS)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PHYSICS)
    real_hist = field_histogram(real)
    gen_hist = field_histogram(generated)
    pk = power_spectrum_summary(real, generated, nbins=25)
    nn = nearest_neighbor_distances(
        evenly_limit(bundle["real"], MAX_REAL_NN),
        evenly_limit(bundle["generated"], MAX_GENERATED_NN),
        max_real=MAX_REAL_NN,
        max_generated=MAX_GENERATED_NN,
    )
    physics_rows.append({
        "run_base": base,
        "label": bundle["spec"]["label"],
        "n_real_loaded": len(bundle["real"]),
        "n_generated_loaded": len(bundle["generated"]),
        "n_real_used": len(real),
        "n_generated_used": len(generated),
        "real_min": real_hist["min"],
        "generated_min": gen_hist["min"],
        "real_max": real_hist["max"],
        "generated_max": gen_hist["max"],
        "real_mean": real_hist["mean"],
        "generated_mean": gen_hist["mean"],
        "real_std": real_hist["std"],
        "generated_std": gen_hist["std"],
        "real_frac_abs_ge_0999": real_hist["frac_abs_ge_0999"],
        "generated_frac_abs_ge_0999": gen_hist["frac_abs_ge_0999"],
        **pk,
        **nn,
    })

physics_df = pd.DataFrame(physics_rows).sort_values("run_base")
physics_csv = OUTPUT_DIR / "normalization_fixes_physics_metrics.csv"
physics_df.to_csv(physics_csv, index=False)
display(physics_df)
print("wrote", physics_csv)


In [ ]:
n = len(data)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 8), squeeze=False)

for col, (base, bundle) in enumerate(data.items()):
    real = evenly_limit(bundle["real"], MAX_REAL_PHYSICS)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PHYSICS)

    real_hist = field_histogram(real)
    gen_hist = field_histogram(generated)
    edges = np.asarray(real_hist["bin_edges"])
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax = axes[0, col]
    ax.plot(centers, real_hist["hist"], color="black", label="real")
    ax.plot(centers, gen_hist["hist"], color="tab:blue", label="generated")
    ax.set_yscale("log")
    ax.set_title(bundle["spec"]["label"])
    ax.set_xlabel("normalized field value")
    ax.set_ylabel("density")
    ax.legend()

    pk_real, kbins = batch_power_spectra(real, nbins=25)
    pk_gen, _ = batch_power_spectra(generated, nbins=25)
    ratio = np.nanmean(pk_gen, axis=0) / np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
    ax = axes[1, col]
    ax.plot(kbins, ratio, marker="o", color="tab:blue")
    ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5)
    ax.set_xlabel("k bin")
    ax.set_ylabel("generated P(k) / real P(k)")
    ax.set_ylim(0, max(2.0, np.nanmax(ratio) * 1.1))

fig.tight_layout()
physics_fig = OUTPUT_DIR / "normalization_fixes_physics_diagnostics.png"
fig.savefig(physics_fig, dpi=180)
print("wrote", physics_fig)

## Power-Spectrum Overlay

Mean real and generated radial power spectra plus generated/real ratio. This is the main physics-fidelity diagnostic; SSCD answers a different near-copy question.


In [ ]:
fig, axes = plt.subplots(2, len(data), figsize=(5 * len(data), 8), squeeze=False)
for col, (base, bundle) in enumerate(data.items()):
    real = evenly_limit(bundle["real"], MAX_REAL_PHYSICS)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PHYSICS)
    pk_real, kbins = batch_power_spectra(real, nbins=25)
    pk_gen, _ = batch_power_spectra(generated, nbins=25)
    real_mean = np.nanmean(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    ratio = gen_mean / np.clip(real_mean, 1e-30, None)

    ax = axes[0, col]
    ax.plot(kbins, real_mean, marker="o", color="black", label="real")
    ax.plot(kbins, gen_mean, marker="o", color="tab:blue", label="generated")
    ax.set_yscale("log")
    ax.set_title(bundle["spec"]["label"])
    ax.set_xlabel("k bin")
    ax.set_ylabel("mean P(k)")
    ax.grid(alpha=0.25)
    ax.legend()

    ax = axes[1, col]
    ax.plot(kbins, ratio, marker="o", color="tab:blue")
    ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5)
    ax.set_xlabel("k bin")
    ax.set_ylabel("generated / real")
    ax.grid(alpha=0.25)

fig.tight_layout()
pk_overlay_fig = OUTPUT_DIR / "normalization_fixes_pk_overlay.png"
fig.savefig(pk_overlay_fig, dpi=180)
print("wrote", pk_overlay_fig)


## SSCD Near-Copy / Generalization Evaluation

Run this on a GPU node for the full comparison. The caps above make the notebook safe for an interactive smoke test.

In [ ]:
if not SSCD_PATH.exists():
    raise FileNotFoundError(f"SSCD checkpoint not found: {SSCD_PATH}")

sscd_model = load_sscd_torchscript(SSCD_PATH, device=DEVICE)
print("loaded SSCD on", DEVICE)

In [ ]:
sscd_rows = []
embedding_cache: dict[str, dict[str, torch.Tensor]] = {}

for base, bundle in data.items():
    real = evenly_limit(bundle["real"], MAX_REAL_SSCD)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_SSCD)
    print(f"{base}: SSCD generated={len(generated)} real={len(real)}")

    gen_emb = sscd_embeddings(
        generated,
        sscd_model,
        device=DEVICE,
        batch_size=SSCD_BATCH_SIZE,
        render_mode=SSCD_RENDER_MODE,
    )
    real_emb = sscd_embeddings(
        real,
        sscd_model,
        device=DEVICE,
        batch_size=SSCD_BATCH_SIZE,
        render_mode=SSCD_RENDER_MODE,
    )
    embedding_cache[base] = {"generated": gen_emb, "real": real_emb}
    metrics = sscd_generalization_metrics(gen_emb, real_emb, threshold=SSCD_THRESHOLD)
    sscd_rows.append({
        "run_base": base,
        "label": bundle["spec"]["label"],
        "n_generated_sscd": len(generated),
        "n_real_sscd": len(real),
        "render_mode": SSCD_RENDER_MODE,
        **metrics,
    })

sscd_df = pd.DataFrame(sscd_rows).sort_values("run_base")
sscd_csv = OUTPUT_DIR / "normalization_fixes_sscd_metrics.csv"
sscd_df.to_csv(sscd_csv, index=False)
display(sscd_df)
print("wrote", sscd_csv)

In [ ]:
sweep_rows = []
for base, cache in embedding_cache.items():
    for threshold in SSCD_THRESHOLDS:
        metrics = sscd_generalization_metrics(cache["generated"], cache["real"], threshold=threshold)
        sweep_rows.append({"run_base": base, "threshold": threshold, **metrics})

threshold_df = pd.DataFrame(sweep_rows)
threshold_csv = OUTPUT_DIR / "normalization_fixes_sscd_threshold_sweep.csv"
threshold_df.to_csv(threshold_csv, index=False)
display(threshold_df.pivot(index="run_base", columns="threshold", values="generalization_score"))
print("wrote", threshold_csv)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for base, group in threshold_df.groupby("run_base"):
    group = group.sort_values("threshold")
    ax.plot(group["threshold"], group["generalization_score"], marker="o", label=base)
ax.set_xlabel("SSCD copy threshold")
ax.set_ylabel("generalization score")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
sscd_fig = OUTPUT_DIR / "normalization_fixes_sscd_threshold_sweep.png"
fig.savefig(sscd_fig, dpi=180)
print("wrote", sscd_fig)

## Combined Summary

In [ ]:
combined = physics_df.merge(
    sscd_df[["run_base", "generalization_score", "copy_fraction", "max_similarity_median", "max_similarity_q99"]],
    on="run_base",
    how="left",
)
combined_csv = OUTPUT_DIR / "normalization_fixes_combined_summary.csv"
combined.to_csv(combined_csv, index=False)
display(combined)
print("wrote", combined_csv)